# Execution guide

Read the lessons in order. Python labs are executable. Java and JavaScript examples include Python launch cells that use your installed JDK or Node.js; this is a Python-kernel notebook, not a Java kernel. Provider-backed integration recipes remain displayed text and are not run. The React project and Spring project have separate build instructions in START-HERE.md. Recorded outputs came from the accompanying validation run. To rerun, select an environment with the versions in requirements-tested.txt.

# 19 — Seeing failures and testing the whole experience

## 1. Three ways to understand a busy kitchen

A log is a note: “order 17 failed.” A metric is a number over time: “three orders failed this minute.” A trace is the journey of one order through cashier, cook and delivery. Use each for the question it answers. Logging every token is neither necessary nor privacy-friendly.

Study Coach creates a Java `job.execute` span and propagates W3C trace context in the HTTP request to Python. Python creates a child request span. The integration checks matching trace IDs and the child's parent span ID. This proves a cross-service link; it does not prove that every background message and browser action is instrumented.

JSON trace files provide an offline inspection path. Optional OTLP HTTP export sends spans to a collector. Java Micrometer and Python Prometheus expose counters and durations. Keep these endpoints protected: operational metadata can reveal internals even when it contains no passwords.

The monitoring integration ran native Prometheus 3.14.0, scraped the protected Python endpoint, queried the counter for a real completed answer, and verified a timed alert fixture using promtool. Open the [saved monitoring dashboard](labs/study-coach/monitoring-dashboard.html). Its values are explicitly a recorded run, not current service health. The later collector lab also ran native OpenTelemetry Collector 0.161.0, verified its archive digest, validated its configuration and received an actual Python-exported OTLP trace. See `labs/collector-lab/report.json`. This proves the local receive/debug-export path, not Grafana, durable trace storage, TLS/authentication or outage buffering.

## 2. Labels can eat your memory

A metric with labels `route` and `status` has a manageable number of combinations. Adding a unique user ID or question to every label can create millions of time series. That is high cardinality. Put bounded categories in metrics and controlled diagnostic IDs in traces or logs. Never use raw prompts, credentials or bearer tokens as metric labels.

The Python middleware groups unknown paths into `other`. It records counts and duration for `/answer`, `/health` and `/metrics`. The Java worker records duration and failure counts. The trace files contain identifiers and timing, not the question text or authorization header.

## 3. Average is a quiet liar

If nine requests take 100 milliseconds and one takes 10 seconds, the average is 1.09 seconds. Most users waited 0.1 seconds; one waited 10 seconds. Percentiles show the distribution. For a small nearest-rank sample, p95 means the value at rank ceil(0.95 × n) in sorted order. With only 12 samples that is the maximum, so it is a noisy estimate.



In [1]:
# lab: latency_percentiles
import math,statistics
samples=[100]*9+[10000]
def percentile(xs,p):
    if not xs or not 0<p<=1:raise ValueError('invalid sample or percentile')
    return sorted(xs)[math.ceil(p*len(xs))-1]
assert percentile(samples,.5)==100 and percentile(samples,.95)==10000
print({'mean_ms':statistics.mean(samples),'p50_ms':percentile(samples,.5),'p95_ms':percentile(samples,.95)})


{'mean_ms': 1090, 'p50_ms': 100, 'p95_ms': 10000}



The distributed test measures 12 jobs with four concurrent clients. Its throughput and latency are observations on this machine, not a production capacity guarantee. A serious load study needs warm-up, realistic request distributions, independent load generation, long steady-state runs, resource saturation measurements and repeated trials.

## 4. Alerts should describe user pain

A service-level indicator is a measured property such as successful completed jobs divided by valid submitted jobs. A service-level objective is a target over a specified window. Define what counts as success and what exclusions apply. A job that returns HTTP 200 but later fails should not disappear from your success measurement.

An error budget is the allowed bad fraction. At 99.9% success, the budget is 0.1%. If the current failure fraction is 1%, the service is consuming that allowance at ten times the target rate. Alert over both short and long windows to avoid reacting to a single noisy minute while still catching rapid damage.



In [2]:
# lab: error_budget_burn
def burn(failures,total,target=.999):
    if total<=0 or not 0<target<1 or not 0<=failures<=total:raise ValueError('invalid inputs')
    return (failures/total)/(1-target)
assert abs(burn(10,1000)-10)<1e-10
print('1% errors burns a 99.9% success budget at approximately 10x.')


1% errors burns a 99.9% success budget at approximately 10x.



## 5. Browser engines are not interchangeable evidence

Chromium, Firefox and WebKit use different implementations. Test your central journeys in all three. WebKit running under Playwright on Windows is useful engine coverage; it is not a test on a physical iPhone or the shipping Safari application. State that distinction in a release report.

The project tests sign-in, job creation, approval, streamed completion, cancellation and navigation. It runs axe rules on the completed page and exercises keyboard interactions. Add cases for a rejected login, missing field, expired session, interrupted stream, route reload and cancellation during work as the product evolves.

## 6. Accessibility beyond an automated score

Use a keyboard from the first control through the final action. Focus must be visible, order sensible and dialogs escapable. A screen reader needs accessible names and meaningful status announcements. At enlarged text and narrow widths, content must remain readable without losing actions. Avoid conveying state only through red and green.

Automated rules can detect many missing labels and contrast failures. They cannot establish that every instruction is understandable, that focus moves helpfully after every dynamic update, or that a whole workflow works with a screen reader. Record manual checks separately from automated results. Do not write “WCAG certified” from a zero-violation scan.

## 7. Failure drills

Drill one: stop the broker. Expected: HTTP approval commits to the outbox; work remains pending; a visible status explains the wait; restart permits delivery. Drill two: stop a worker after claiming. Expected: the lease expires; another worker claims; the old token cannot write. Drill three: disconnect a model stream. Expected: no `done` state without a final completion marker. Drill four: remove permission. Expected: new unauthorized requests fail, with session revocation behavior matching the documented design.

For every drill, write the hypothesis before the test. Record the observed outcome and recovery time. A failed hypothesis is useful evidence to fix the design, not a reason to hide the test.

## 8. Debugging interview

Question: “The p99 doubled but CPU is low. What next?” Answer: compare traces for time waiting on database connections, locks, network calls, queue delay and downstream services. Low CPU is consistent with blocked work. Inspect saturation of the constrained resource rather than guessing that more application replicas will help.

Question: “Why is the graph expensive after adding user labels?” Answer: each distinct label combination creates another series. Remove unbounded labels, aggregate metrics by meaningful bounded dimensions and retain per-request correlation in traces with a privacy policy.

Reference provenance: the native monitoring binary and checksum came from the [official Prometheus download page](https://prometheus.io/download/). The included setup script pins the version and checksum. This reference is optional; the concepts, test and interpretation are provided here.
